In [1]:
# pip install langchain
# pip install langchain_openai
# pip install langchain_core
# pip install langchain_huggingface
import langchain
import langchain_openai
import langchain_core

print('version of langchain: ',langchain.__version__)
print('version of langchain_openai: ',langchain_openai.__version__)
print('version of langchain_core: ',langchain_core.__version__)

version of langchain:  1.3.14
version of langchain_openai:  1.4.1
version of langchain_core:  1.5.1


In [2]:
from dotenv import load_dotenv
load_dotenv()


True

In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4")

In [4]:
output = llm.invoke("Hello, how are you?")

In [5]:
output.content

"As an artificial intelligence, I don't have feelings, but I'm here and ready to assist you. How can I help you today?"

In [6]:
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint

In [7]:
hf_llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-72B-Instruct",
    provider = 'novita'
)

llm = ChatHuggingFace(llm=hf_llm)

/Users/rahultiwari/Documents/02_Freelancing/Rbyte_project/Sathish_AI/test-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
llm.invoke("Hello, how are you?").content

"Hello! I'm an AI and I don't have feelings, but I'm here to help you. How can I assist you today?"

In [7]:
from langchain_core.prompts import ChatPromptTemplate


simple_template = ChatPromptTemplate.from_template(
    'Reply in one sentence: {questions}'
)

prompt = simple_template.invoke({"questions": "What is the capital of France?" })


In [8]:
simple_template.messages

[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['questions'], input_types={}, partial_variables={}, template='Reply in one sentence: {questions}'), additional_kwargs={})]

In [9]:
prompt

ChatPromptValue(messages=[HumanMessage(content='Reply in one sentence: What is the capital of France?', additional_kwargs={}, response_metadata={})])

In [10]:
from langchain_core.output_parsers import StrOutputParser
str_parser = StrOutputParser()
chain = llm | str_parser

In [11]:
chain.invoke('hi')

'Hello! How can I assist you today?'

In [12]:
classifier_template = ChatPromptTemplate.from_messages([
    ('system', 'You are an expert support email classifier for a B2B SaaS company.'),
    ('human',  'Classify this email.\nSubject: {subject}\nBody: {body}\n\nReturn ONLY: Category | Urgency')
])


In [13]:
chain = classifier_template | llm | str_parser

In [14]:
email = {
    'subject': "Can't login — paid for annual plan last week",
    'body'   : "Our team can't login. We paid last week. Board demo in 3 hours."
}

chain.invoke(email)

'Category: Login Issues | Urgency: High'

In [15]:
emails = [
    {'subject': 'Payment failed',   'body': 'Charged twice this month.'},
    {'subject': 'App crash',        'body': 'Crashes on Settings page.'},
    {'subject': 'Add Slack?',       'body': 'Want Slack notifications.'},
    {'subject': 'WIN FREE IPHONE',  'body': 'Click here NOW!!!'},
    {'subject': 'Invoice question', 'body': 'Can I get annual invoice?'},
]

import time
start = time.time()
loop_result = []
for email in emails:
    loop_result.append(chain.invoke(email))

loop_time = time.time()-start
print(loop_time)

12.02941632270813


In [16]:
batch_results = chain.batch(emails,
                                    config={'max_concurrency':3})

In [17]:
batch_results

['Category: Billing/Payment Issues | Urgency: High',
 'Category: Technical Issues - App Functionality | Urgency: High',
 'Category: Feature Request | Urgency: Low',
 'Category: Spam | Urgency: Low',
 'Category: Billing/Invoicing | Urgency: Normal']

In [18]:
# chain.with_retry(
#     stop_after_attempt=3,
#     wait_exponential_jitter=True
# )

In [25]:
# emails ---> spam --> mislead --> I dont want to pass to my llm
SPAM_KEYWORDS = ["lottery", "winner", "click here", "free money", "urgent", "congratulations", "prize"]

def validate_email(email):
    print("before parsing email: ",email['body'])
    body = email.get("body","").lower()
    print("after parsing email: ",email['body'])
    found_spam = [word for word in SPAM_KEYWORDS if word in body]
    print("found_spam",found_spam)
    if found_spam:
        raise ValueError("Spam word found:",found_spam)
    return email

# validate_email({"body": "Congratulations! You are a lottery winner"})

In [24]:
chain = validate_email| classifier_template | llm | str_parser
chain.invoke( {'subject': 'Payment failed',   'body': 'Charged twice this month.'})

before parsing email:  Charged twice this month.
after parsing email:  Charged twice this month.
found_spam []


'Category: Billing/Payment Issues | Urgency: High'

In [ ]:
outputs_from_llm = [
    'CATEGORY: Billing\nURGENCY: High',        # normal
    'CATEGORY:  Billing\nURGENCY: High',       # extra space!
    'Category: Billing\nUrgency: High',         # different case!
    'CATEGORY: Billing (payment issue)\nURGENCY: High',  # extra text!
    'I think this is CATEGORY: Billing\nURGENCY: High',  # explanation first!
]

In [26]:
from pydantic import BaseModel, Field
from typing import Literal


In [27]:
class EmailClassification(BaseModel):
    category : Literal['Billing','technical','Feature request','Other'] = Field(description='the category of the support email')
    urgency: Literal['High', 'Medium', 'Low'] = Field(
        description='Urgency based on business impact'
    )

    confidence: int = Field(
        description='Confidence score from 1 to 10',
        ge=1,   # ge = greater than or equal (Pydantic v2!)
        le=10   # le = less than or equal
    )

    reasoning: str = Field(
        description='One sentence explanation of the classification'
    )

In [30]:
from langchain_core.output_parsers import PydanticOutputParser
parser = PydanticOutputParser(pydantic_object=EmailClassification)


In [32]:
# print(parser.get_format_instructions())

In [33]:
cot_prompt = ChatPromptTemplate.from_messages([
    ('system', '''You are an expert support email classifier.

Classification Rules:
- Login broken after payment → Billing (NOT Technical)
- App crashes → Technical
- Pricing complaints + evaluating alternatives → Churn Risk
- Feature requests → Feature Request

Think step by step before classifying.

{format_instruction}'''),
     ('human', 'Subject: {subject}\nBody: {body}')
]).partial(format_instruction= parser.get_format_instructions())

In [34]:
cot_prompt

ChatPromptTemplate(input_variables=['body', 'subject'], input_types={}, partial_variables={'format_instruction': 'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"category": {"description": "the category of the support email", "enum": ["Billing", "technical", "Feature request", "Other"], "title": "Category", "type": "string"}, "urgency": {"description": "Urgency based on business impact", "enum": ["High", "Medium", "Low"], "title": "Urgency", "type": "string"}, "confidence": {"description": "Confidence score from 1 to 10", "maximum": 10, "minimum": 1, "title": "Confid

In [37]:
cot_chain = cot_prompt | llm | parser
email = {
    'subject': "Can't login — paid for annual plan last week",
    'body': 'Our entire team cannot login. We paid for the annual plan '
            'last week. We have a board demo in 3 hours!'
}
result = cot_chain.invoke(email)

In [39]:
result.category,result.urgency,result.confidence,result.reasoning

('Billing',
 'High',
 9,
 'The issue is with login after a recent payment, indicating an issue in billing rather than a technical one.')